# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response
- `SUBSET_SIZE` — `None` = all questions; or a small integer for a quick smoke test
- `GREEDY` — `True` → `temperature=0` (deterministic single sample)
- `SEED` — optional RNG seed for `SamplingParams` (`None` to omit)
- `MCQ_SELF_CONSISTENCY_N` — `1` = one sample; `>1` = multiple samples for **MCQ only**, then majority vote on the letter inside `\boxed{}`
- `GPU_MEMORY_UTILIZATION`, `MAX_MODEL_LEN`, `MAX_NUM_SEQS`, `MAX_NUM_BATCHED_TOKENS` — vLLM memory / batching (lower if you OOM)

In [ ]:
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Any, Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768
# Smoke test: set to a small int (e.g. 5). Full public eval: None
SUBSET_SIZE: int | None = None

GREEDY = False                     # True: temperature 0, single-sample decoding
SEED: Optional[int] = 42           # None: let vLLM default
MCQ_SELF_CONSISTENCY_N = 8         # >1: extra MCQ samples + majority vote
FREEFORM_SELF_CONSISTENCY_N = 8    # >1: extra free-form samples + majority vote on boxed answer

GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 32768
MAX_NUM_SEQS = 256
MAX_NUM_BATCHED_TOKENS = 65536

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician who solves problems with rigorous step-by-step reasoning.\n\n"
    "Instructions:\n"
    "1. Break the problem into clear steps. Show all work.\n"
    "2. For each step, state what you are doing and why.\n"
    "3. Double-check your arithmetic and algebra before concluding.\n"
    "4. Put your final answer inside \\boxed{}.\n"
    "5. If the problem asks for multiple sub-answers, separate them by commas "
    "inside a single \\boxed{}, e.g. \\boxed{3, 7}.\n"
    "6. Give exact answers (fractions, radicals, pi) unless a decimal approximation is explicitly requested.\n"
    "7. If the answer is a well-known constant or expression, simplify it fully."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician who solves problems with rigorous step-by-step reasoning.\n\n"
    "Instructions:\n"
    "1. First, solve the problem yourself step-by-step WITHOUT looking at the options.\n"
    "2. Then compare your solution to each option to find the match.\n"
    "3. If your answer does not match any option, re-examine your work and "
    "try alternative approaches.\n"
    "4. Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user_prompt = (
            f"{question}\n\n"
            f"Options:\n{opts_text}\n\n"
            f"Solve the problem step-by-step, then select the correct option letter."
        )
        return SYSTEM_PROMPT_MCQ, user_prompt
    user_prompt = (
        f"{question}\n\n"
        f"Solve this problem step-by-step, showing all work. "
        f"Put your final answer in \\boxed{{}}."
    )
    return SYSTEM_PROMPT_MATH, user_prompt


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 300 chars) ──")
    print(usr_p[:300], "...\n")

## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters (tune in the config cell): `GPU_MEMORY_UTILIZATION`, `MAX_MODEL_LEN`, `MAX_NUM_SEQS`, `MAX_NUM_BATCHED_TOKENS`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=MAX_NUM_SEQS,
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
)

print("Model loaded. Sampling params are built in the next section.")

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()`.

- **Default** — one forward pass over all prompts with `n=1`.
- **`MCQ_SELF_CONSISTENCY_N > 1`** — MCQ prompts use `n>1` (stochastic samples); we keep **one** response string per question by majority vote on the extracted `\boxed{}` letter. Mixed MCQ + free-form runs **two** passes (MCQ with `n>1`, free-form with `n=1`) so free-form cost stays unchanged.
- **`GREEDY`** — `temperature=0`. If you also set `MCQ_SELF_CONSISTENCY_N > 1`, the MCQ pass temporarily uses stochastic decoding so samples are not identical.

In [ ]:
def extract_letter(text: str) -> str:
    """Extract a single letter answer from \\boxed{} or fallback to last capital letter."""
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def extract_boxed_answer(text: str) -> str:
    """Extract the raw content inside the last \\boxed{} in text."""
    # Strip thinking tags
    think_end = text.rfind("</think>")
    search_text = text[think_end + len("</think>"):] if think_end >= 0 else text

    idx = search_text.rfind("\\boxed{")
    if idx < 0:
        return ""
    brace_start = idx + len("\\boxed{")
    depth = 1
    i = brace_start
    while i < len(search_text) and depth > 0:
        if search_text[i] == '{':
            depth += 1
        elif search_text[i] == '}':
            depth -= 1
        i += 1
    if depth == 0:
        return search_text[brace_start:i - 1].strip()
    return ""


def pick_mcq_majority_text(completion_outputs: list[Any]) -> str:
    """Return one completion text: majority vote on boxed letter; tie -> first winning sample."""
    outs = list(completion_outputs)
    letters = [extract_letter(o.text.strip()) for o in outs]
    valid = [L for L in letters if L]
    if not valid:
        return outs[0].text.strip()
    winner, _ = Counter(valid).most_common(1)[0]
    for o, L in zip(outs, letters):
        if L == winner:
            return o.text.strip()
    return outs[0].text.strip()


def pick_freeform_majority_text(completion_outputs: list[Any]) -> str:
    """Return one completion text: majority vote on boxed answer string for free-form."""
    outs = list(completion_outputs)
    answers = [extract_boxed_answer(o.text.strip()) for o in outs]
    valid = [(ans, i) for i, ans in enumerate(answers) if ans]
    if not valid:
        return outs[0].text.strip()

    # Normalize whitespace for comparison
    normalized = [re.sub(r"\s+", " ", a).strip().lower() for a, _ in valid]
    counter = Counter(normalized)
    winner_norm, _ = counter.most_common(1)[0]

    for (ans, idx), norm in zip(valid, normalized):
        if norm == winner_norm:
            return outs[idx].text.strip()
    return outs[0].text.strip()


def make_sampling_params(*, n: int, force_stochastic: bool) -> SamplingParams:
    """Build vLLM sampling params. When ``force_stochastic``, ignore ``GREEDY`` (for n>1)."""
    if GREEDY and not force_stochastic:
        temperature = 0.0
        top_p = 1.0
        top_k = -1
    else:
        temperature = 0.7
        top_p = 0.95
        top_k = 40
    kwargs: dict[str, Any] = {
        "max_tokens": MAX_TOKENS,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "min_p": 0.0,
        "presence_penalty": 0.0,
        "repetition_penalty": 1.0,
        "n": n,
    }
    if SEED is not None:
        kwargs["seed"] = SEED
    return SamplingParams(**kwargs)


work_data = data[:SUBSET_SIZE] if SUBSET_SIZE is not None else data
mcq_idx = [i for i, item in enumerate(work_data) if item.get("options")]
free_idx = [i for i, item in enumerate(work_data) if not item.get("options")]

if MCQ_SELF_CONSISTENCY_N < 1:
    raise ValueError("MCQ_SELF_CONSISTENCY_N must be >= 1")
if FREEFORM_SELF_CONSISTENCY_N < 1:
    raise ValueError("FREEFORM_SELF_CONSISTENCY_N must be >= 1")

mcq_n = MCQ_SELF_CONSISTENCY_N
free_n = FREEFORM_SELF_CONSISTENCY_N


def prompt_for_item(item: dict) -> str:
    """Format a single item into a full chat-template prompt string."""
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )


responses = [""] * len(work_data)

# ── MCQ pass ──────────────────────────────────────────────────────────────────
if mcq_idx:
    mcq_prompts = [prompt_for_item(work_data[i]) for i in mcq_idx]
    force_stoch = GREEDY and mcq_n > 1
    sp_mcq = make_sampling_params(n=mcq_n, force_stochastic=force_stoch)
    print(f"Generating {len(mcq_prompts)} MCQ with n={mcq_n} ...")
    out_mcq = llm.generate(mcq_prompts, sampling_params=sp_mcq)
    for local_i, global_i in enumerate(mcq_idx):
        if mcq_n > 1:
            responses[global_i] = pick_mcq_majority_text(out_mcq[local_i].outputs)
        else:
            responses[global_i] = out_mcq[local_i].outputs[0].text.strip()

# ── Free-form pass ────────────────────────────────────────────────────────────
if free_idx:
    free_prompts = [prompt_for_item(work_data[i]) for i in free_idx]
    force_stoch = GREEDY and free_n > 1
    sp_free = make_sampling_params(n=free_n, force_stochastic=force_stoch)
    print(f"Generating {len(free_prompts)} free-form with n={free_n} ...")
    out_free = llm.generate(free_prompts, sampling_params=sp_free)
    for local_i, global_i in enumerate(free_idx):
        if free_n > 1:
            responses[global_i] = pick_freeform_majority_text(out_free[local_i].outputs)
        else:
            responses[global_i] = out_free[local_i].outputs[0].text.strip()

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={work_data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(work_data, responses), total=len(responses), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!